# SEM.9.7 — Notebook 1: Annual low-flow channel-water masks (Google Earth Engine)

Reproducibility notebook for **SFFS technical report SEM.9.7, *Geomorphic Evolution of the Lower Semliki: Bed and Bank Change, 2000–2026*** (Ministry of Water and Environment, Uganda, 2026), Section 4.1.

For every year it builds one composite of all clear scenes of the **low-flow season** (15 December – 28 February, falling back to June–August when that season is cloudy) and classifies water as **MNDWI above an Otsu threshold** fitted inside a 1,500 m corridor around the river. Comparing the same season every year avoids the false "erosion" that appears when a wet-season mask is compared with a dry one. Two bands are written: `water` (0/1) and `valid` (0 where every scene of the season was cloudy).

| Sensor | Collection | Years | Pixel |
|---|---|---|---|
| Landsat 5 TM, 7 ETM+ | `LANDSAT/LT05`, `LANDSAT/LE07` Collection 2 Level 2 | 2000–2015 | 30 m |
| Landsat 8, 9 OLI | `LANDSAT/LC08`, `LANDSAT/LC09` Collection 2 Level 2 | 2016–2026 | 30 m |
| Sentinel-2 MSI | `COPERNICUS/S2_HARMONIZED` (L1C) + `S2_CLOUD_PROBABILITY` | 2016–2026 | 10 m |

**This notebook needs a Google Earth Engine account** (`pip install earthengine-api`, then `earthengine authenticate`). Its output — the masks used in the report — is already in `data/masks/` as `.npz` files (`water`, `valid`, `transform` = [a, b, c, d, e, f] of the UTM 35N grid, EPSG:32635), so Notebooks 2 and 3 run without it. It is shown here un-run.

Contains modified Copernicus Sentinel data (2016–2026) and Landsat imagery courtesy of the U.S. Geological Survey.

In [ ]:
import os, zipfile
if not os.path.isdir("data"):      # the masks and published results ship as one zip beside the notebooks
    zipfile.ZipFile("SEM.9.7_data_masks_and_results.zip").extractall(".")

In [ ]:
import os, sys, json, math, csv
import numpy as np
import ee

GEE_PROJECT = None                     # your Earth Engine Cloud project id, or None for the saved default
SENSOR = "landsat"                     # "landsat" or "s2"
START, END = 2000, 2026
SEASON, FALLBACK = "12-15,02-28", "06-01,08-31"
BUFFER_M = 1500
OUT_DIR = os.path.abspath("outputs/masks_%s" % SENSOR)

ee.Initialize(project=GEE_PROJECT) if GEE_PROJECT else ee.Initialize()
CRS = "EPSG:32635"; SCALE = {"s2": 10, "landsat": 30}[SENSOR]

## The corridor
The river centreline is the published baseline (`data/results/landsat_2000_2026/transects.csv`, UTM 35N), converted to WGS 84 and simplified to every 10th point.

In [ ]:
from pyproj import Transformer
rows = list(csv.DictReader(open("data/results/landsat_2000_2026/transects.csv")))
t = Transformer.from_crs(32635, 4326, always_xy=True)
lonlat = [list(t.transform(float(r["x"]), float(r["y"]))) for r in rows[::10]]
cl = ee.Geometry.LineString(lonlat)
AOI = cl.buffer(BUFFER_M)

## Image collections

In [ ]:
def s2_collection(t0, t1):
    s2 = ee.ImageCollection("COPERNICUS/S2_HARMONIZED").filterBounds(AOI).filterDate(t0, t1)
    prob = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY").filterBounds(AOI).filterDate(t0, t1)
    j = ee.Join.saveFirst("cp").apply(s2, prob, ee.Filter.equals(leftField="system:index", rightField="system:index"))
    def prep(img):
        img = ee.Image(img)
        clear = ee.Image(img.get("cp")).select("probability").lt(40)
        return img.normalizedDifference(["B3", "B11"]).rename("wi").updateMask(clear).copyProperties(img, ["system:time_start"])
    return ee.ImageCollection(j).map(prep)

def landsat_collection(t0, t1):
    def prep(green, swir):
        def f(img):
            qa = img.select("QA_PIXEL")
            clear = qa.bitwiseAnd(1 << 1).eq(0).And(qa.bitwiseAnd(1 << 3).eq(0)).And(qa.bitwiseAnd(1 << 4).eq(0))
            sr = img.select([green, swir]).multiply(0.0000275).add(-0.2)
            return sr.normalizedDifference([green, swir]).rename("wi").updateMask(clear).copyProperties(img, ["system:time_start"])
        return f
    tm = lambda c: ee.ImageCollection(c).filterBounds(AOI).filterDate(t0, t1).map(prep("SR_B2", "SR_B5"))
    oli = lambda c: ee.ImageCollection(c).filterBounds(AOI).filterDate(t0, t1).map(prep("SR_B3", "SR_B6"))
    return tm("LANDSAT/LT05/C02/T1_L2").merge(tm("LANDSAT/LE07/C02/T1_L2")).merge(oli("LANDSAT/LC08/C02/T1_L2")).merge(oli("LANDSAT/LC09/C02/T1_L2"))

COLLECT = {"s2": s2_collection, "landsat": landsat_collection}[SENSOR]

def season_dates(year, season):
    (m0, d0), (m1, d1) = [tuple(int(v) for v in s.split("-")) for s in season.split(",")]
    y0 = year - 1 if (m0, d0) > (m1, d1) else year
    return "%d-%02d-%02d" % (y0, m0, d0), "%d-%02d-%02d" % (year, m1, d1)

## Otsu threshold inside the corridor, and the yearly mask

In [ ]:
def otsu(img, band="wi"):
    h = ee.Dictionary(img.select(band).reduceRegion(ee.Reducer.histogram(255, 0.01), AOI, SCALE * 2, maxPixels=1e9).get(band))
    counts, means = ee.Array(h.get("histogram")), ee.Array(h.get("bucketMeans"))
    size = means.length().get([0]); total = counts.reduce(ee.Reducer.sum(), [0]).get([0])
    mean = counts.multiply(means).reduce(ee.Reducer.sum(), [0]).get([0]).divide(total)
    def bss(i):
        a_c = counts.slice(0, 0, i); a_n = a_c.reduce(ee.Reducer.sum(), [0]).get([0])
        a_mean = means.slice(0, 0, i).multiply(a_c).reduce(ee.Reducer.sum(), [0]).get([0]).divide(a_n)
        b_n = total.subtract(a_n); b_mean = mean.multiply(total).subtract(a_n.multiply(a_mean)).divide(b_n)
        return a_n.multiply(a_mean.subtract(mean).pow(2)).add(b_n.multiply(b_mean.subtract(mean).pow(2)))
    return means.sort(ee.Array(ee.List.sequence(1, size).map(bss))).get([-1])

def year_mask(year):
    col = COLLECT(*season_dates(year, SEASON))
    col = ee.ImageCollection(ee.Algorithms.If(col.size().gt(0), col, COLLECT(*season_dates(year, FALLBACK))))
    comp = col.median().clip(AOI)
    thr = otsu(comp)
    water = comp.select("wi").gt(ee.Number(thr)).unmask(0).rename("water").toByte()
    valid = comp.select("wi").mask().unmask(0).rename("valid").toByte()
    return water.addBands(valid).set({"year": year, "n_scenes": col.size(), "threshold": thr})

## Download each year as `.npz` on the UTM 35N grid

In [ ]:
def download(img, path):
    b = AOI.transform(CRS, 1).bounds(1, CRS).coordinates().getInfo()[0]
    xs, ys = [p[0] for p in b], [p[1] for p in b]
    x0 = math.floor(min(xs) / SCALE) * SCALE; x1 = math.ceil(max(xs) / SCALE) * SCALE
    y0 = math.floor(min(ys) / SCALE) * SCALE; y1 = math.ceil(max(ys) / SCALE) * SCALE
    W, H = int((x1 - x0) / SCALE), int((y1 - y0) / SCALE)
    water = np.zeros((H, W), np.uint8); valid = np.zeros((H, W), np.uint8); T = 1024
    for r0 in range(0, H, T):
        for c0 in range(0, W, T):
            h_, w_ = min(T, H - r0), min(T, W - c0)
            arr = ee.data.computePixels({"expression": img.select(["water", "valid"]), "fileFormat": "NUMPY_NDARRAY",
                "grid": {"dimensions": {"width": w_, "height": h_},
                         "affineTransform": {"scaleX": SCALE, "shearX": 0, "translateX": x0 + c0 * SCALE,
                                             "shearY": 0, "scaleY": -SCALE, "translateY": y1 - r0 * SCALE}, "crsCode": CRS}})
            water[r0:r0 + h_, c0:c0 + w_] = arr["water"]; valid[r0:r0 + h_, c0:c0 + w_] = arr["valid"]
    np.savez_compressed(path, water=water, valid=valid, transform=np.array([SCALE, 0.0, x0, 0.0, -SCALE, y1]))

os.makedirs(OUT_DIR, exist_ok=True)
for y in range(START, END + 1):
    img = year_mask(y)
    info = ee.Dictionary({"n": img.get("n_scenes"), "thr": img.get("threshold")}).getInfo()
    if not info["n"] or info["thr"] is None:
        print(y, "no usable scenes - skipped"); continue
    download(img, os.path.join(OUT_DIR, "semliki_channel_%s_%d.npz" % (SENSOR, y)))
    print(y, "scenes", info["n"], "threshold %.3f" % info["thr"])